In [2]:
import os
import time
from collections import defaultdict
from typing import Dict, List, Tuple
import numpy as np
from tqdm import tqdm
from numpy.matlib import zeros


# Import the methods and metrics
from tools.ewca import EWCA
from tools.sedmtg import SEDMTG, ProteinNetwork
from tools.mpcc import MPCC
from tools.metrics import (precision_recall_fmeasure, coverage_rate, 
                    accuracy, MMR, jaccard_index, total_score, F_Fitness, FS_fitness)

class ProteinComplexExtractor:
    def __init__(self):
        self.protein_to_id: Dict[str, int] = {}
        self.id_to_protein: Dict[int, str] = {}
        self.known_complexes: List[List[str]] = []  # Store known complexes as lists of protein names
        self.graph_ppi: Dict[str, Dict[str, float]] = {}  # For fitness calculation
    
    def load_known_complexes(self, filepath: str):
        """Load known complexes from file"""
        self.known_complexes = []
        with open(filepath, 'r') as f:
            next(f)  # Skip header
            for line_num, line in enumerate(f, 2):  # Start counting from line 2
                line = line.strip()
                if not line:
                    continue
                    
                parts = line.split('\t')
                if len(parts) < 2:
                    print(f"Line {line_num} ignored - incorrect format: {line}")
                    continue
                    
                proteins = parts[1].split(';')
                self.known_complexes.append(proteins)
    
    def build_ppi_graph(self, network_file: str):
        """Build PPI graph structure for fitness calculation from weighted network file"""
        self.graph_ppi = defaultdict(dict)
        with open(network_file, 'r') as f:
            next(f)
            for line in f:
                if line.strip():
                    parts = line.strip().split()
                    if len(parts) == 3:
                        protein1, protein2, weight = parts[0], parts[1], float(parts[2])
                        self.graph_ppi[protein1][protein2] = weight
                        self.graph_ppi[protein2][protein1] = weight
    
    def generate_complexes(self, ewca_file: str, sedmtg_file: str, output_file: str, metrics_file: str):
        """
        Generate complexes and calculate metrics for each solution.
        Each solution is compared independently against the known complexes.
        """
        start_time = time.time()
        
        all_complexes = []
        metrics_results = []
        
        # Method 1: EWCA with varying ss_threshold (8 solutions)
        print("\nRunning EWCA method...")
        ss_thresholds = np.linspace(0.4, 0.68, 8)
        
        for i, ss_threshold in enumerate(tqdm(ss_thresholds, desc="EWCA progress")):
            ewca = EWCA(ewca_file, ss_threshold)
            
            # Run EWCA and get complexes (modified to return complexes instead of saving)
            ewca.load_interactions()
            ewca.calculate_jaccard_distance()
            ewca.calculate_ecv2_weights()
            core_complexes = ewca.detect_core_complexes()
            complexes = ewca.find_attachments(core_complexes)
            filtered_complexes = ewca.filter_redundant_complexes(complexes)
            
            # Convert to protein names and filter (keep only complexes with ≥3 proteins)
            solution_complexes = [
                [ewca.id_to_protein[pid] for pid in members]
                for members in filtered_complexes.values()
                if len(members) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 1,
                'method': 'EWCA',
                'param': f"ss_threshold={ss_threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                precision, recall, fmeasure = precision_recall_fmeasure(solution_complexes, self.known_complexes, threshold=0.2)
                metrics.update({
                    'fmeasure': fmeasure,
                    'coverage_rate': coverage_rate(solution_complexes, self.known_complexes),
                    'accuracy': accuracy(solution_complexes, self.known_complexes, threshold=0.2),
                    'mmr': MMR(solution_complexes, self.known_complexes),
                    'jaccard': jaccard_index(solution_complexes, self.known_complexes),
                    'total_score': total_score(solution_complexes, self.known_complexes, threshold=0.2)
                })
                
                # Fitness metrics
                fitness_scores = []
                for complex in solution_complexes:
                    fitness_scores.append(F_Fitness(complex, self.graph_ppi))
                
                metrics.update({
                    'avg_fitness': np.mean(fitness_scores) if fitness_scores else 0,
                    'max_fitness': max(fitness_scores) if fitness_scores else 0,
                    'total_fitness': FS_fitness(solution_complexes, self.graph_ppi)
                })
                
                print(f"\nEWCA Solution {i+1} (threshold={ss_threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {fmeasure:.4f}, Avg Fitness: {metrics['avg_fitness']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                    'avg_fitness': 0, 'max_fitness': 0, 'total_fitness': 0
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 1,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Method 2: SEDMTG (8 solutions)
        print("\nRunning SEDMTG method...")
        network = ProteinNetwork.from_weighted_network(sedmtg_file, has_header=True)
        sedmtg = SEDMTG(network, iterations=10)
        
        for i in tqdm(range(8), desc="SEDMTG progress"):
            # Run SEDMTG (modified to return complexes for each iteration)
            protein_complexes = sedmtg.detect_complexes()
            
            # Convert to list format and filter
            solution_complexes = [
                proteins for proteins in protein_complexes.values()
                if len(proteins) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 9,
                'method': 'SEDMTG',
                'param': f"iteration_{i + 1}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                precision, recall, fmeasure = precision_recall_fmeasure(solution_complexes, self.known_complexes)
                metrics.update({
                    'fmeasure': fmeasure,
                    'coverage_rate': coverage_rate(solution_complexes, self.known_complexes),
                    'accuracy': accuracy(solution_complexes, self.known_complexes),
                    'mmr': MMR(solution_complexes, self.known_complexes),
                    'jaccard': jaccard_index(solution_complexes, self.known_complexes),
                    'total_score': total_score(solution_complexes, self.known_complexes)
                })
                
                # Fitness metrics
                fitness_scores = []
                for complex in solution_complexes:
                    fitness_scores.append(F_Fitness(complex, self.graph_ppi))
                
                metrics.update({
                    'avg_fitness': np.mean(fitness_scores) if fitness_scores else 0,
                    'max_fitness': max(fitness_scores) if fitness_scores else 0,
                    'total_fitness': FS_fitness(solution_complexes, self.graph_ppi)
                })
                
                print(f"\nSEDMTG Solution {i+9}:")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {fmeasure:.4f}, Avg Fitness: {metrics['avg_fitness']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                    'avg_fitness': 0, 'max_fitness': 0, 'total_fitness': 0
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 9,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        # Method 3: MPCC with varying filter thresholds (8 solutions)
        print("\nRunning MPCC method...")
        mpcc = MPCC()
        
        # Load interactions once (same file as EWCA)
        mpcc.load_interactions(ewca_file)
        mpcc.remove_false_positives()
        
        # Calculate topology scores and combined weights (done once)
        N = len(mpcc.id_label)
        topo_weights = mpcc.calculate_topology_scores(mpcc.relations, N)
        combined_weights = zeros((N, N))
        
        # Create weight matrix
        weight_matrix = zeros((N, N))
        for (i,j), w in mpcc.weights.items():
            weight_matrix[i,j] = w
        
        # Combine weights
        for i in range(N):
            for j in range(N):
                if i < j and weight_matrix[i,j] > 0 and topo_weights[i,j] > 0:
                    combined_weights[i,j] = 2 * weight_matrix[i,j] * topo_weights[i,j] / (weight_matrix[i,j] + topo_weights[i,j])
                    combined_weights[j,i] = combined_weights[i,j]
        
        # Detect seeds once (they don't depend on the filter threshold)
        seeds = mpcc.detect_seeds(list(mpcc.relations.keys()), mpcc.relations, combined_weights)
        
        # Vary the filter threshold from 0.1 to 0.8 in 8 steps
        filter_thresholds = np.linspace(0.1, 0.8, 8)
        
        for i, threshold in enumerate(tqdm(filter_thresholds, desc="MPCC progress")):
            # Identify complexes with current filter threshold
            complexes = mpcc.identify_complexes(seeds, mpcc.relations, combined_weights)
            
            # Calculate scores
            complex_scores = {}
            final_complexes = defaultdict(list)
            count = 1
            
            for cid in list(complexes.keys()):
                if len(complexes[cid]) >= 3:
                    final_complexes[count] = complexes[cid]
                    complex_scores[count] = mpcc.graph_entropy(complexes[cid], mpcc.relations, combined_weights)
                    count += 1
            
            # Filter with current threshold
            filtered = mpcc.filter_redundant(final_complexes, threshold=threshold)
            
            # Convert to protein names
            solution_complexes = [
                [mpcc.id_label[pid] for pid in members]
                for members in filtered.values()
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 17,  # Starts after EWCA (8) and SEDMTG (8)
                'method': 'MPCC',
                'param': f"filter_threshold={threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                precision, recall, fmeasure = precision_recall_fmeasure(solution_complexes, self.known_complexes)
                metrics.update({
                    'fmeasure': fmeasure,
                    'coverage_rate': coverage_rate(solution_complexes, self.known_complexes),
                    'accuracy': accuracy(solution_complexes, self.known_complexes),
                    'mmr': MMR(solution_complexes, self.known_complexes),
                    'jaccard': jaccard_index(solution_complexes, self.known_complexes),
                    'total_score': total_score(solution_complexes, self.known_complexes)
                })
                
                # Fitness metrics
                fitness_scores = []
                for complex in solution_complexes:
                    fitness_scores.append(F_Fitness(complex, self.graph_ppi))
                
                metrics.update({
                    'avg_fitness': np.mean(fitness_scores) if fitness_scores else 0,
                    'max_fitness': max(fitness_scores) if fitness_scores else 0,
                    'total_fitness': FS_fitness(solution_complexes, self.graph_ppi)
                })
                
                print(f"\nMPCC Solution {i+17} (threshold={threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {fmeasure:.4f}, Avg Fitness: {metrics['avg_fitness']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                    'avg_fitness': 0, 'max_fitness': 0, 'total_fitness': 0
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 17,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Save results
        self._save_results(all_complexes, output_file)
        self._save_metrics(metrics_results, metrics_file)
        
        elapsed_time = time.time() - start_time
        print(f"\nTotal processing time: {elapsed_time:.2f} seconds")
        print(f"Complexes saved to {output_file}")
        print(f"Metrics saved to {metrics_file}")
    
    def _save_results(self, all_complexes: List[Dict], output_file: str):
        """Save all complexes to output file in the required format"""
        with open(output_file, 'w') as f:
            f.write("SolutionID\tComplexID\tProteins\n")
            for complex_data in all_complexes:
                f.write(f"{complex_data['solution_id']}\t{complex_data['complex_id']}\t{' '.join(complex_data['proteins'])}\n")
    
    def _save_metrics(self, metrics_results: List[Dict], metrics_file: str):
        """Save metrics to a TSV file with additional information"""
        with open(metrics_file, 'w') as f:
            # Write header
            f.write("SolutionID\tMethod\tParameters\tDetectedComplexes\tKnownComplexes\t"
                    "F-measure\tCoverageRate\tAccuracy\tMMR\tJaccard\tTotalScore\t"
                    "AvgFitness\tMaxFitness\tTotalFitness\n")
            
            # Write data
            for result in metrics_results:
                f.write(
                    f"{result['solution_id']}\t"
                    f"{result['method']}\t"
                    f"{result['param']}\t"
                    f"{result['detected_complexes']}\t"
                    f"{result['known_complexes']}\t"
                    f"{result['fmeasure']:.4f}\t"
                    f"{result['coverage_rate']:.4f}\t"
                    f"{result['accuracy']:.4f}\t"
                    f"{result['mmr']:.4f}\t"
                    f"{result['jaccard']:.4f}\t"
                    f"{result['total_score']:.4f}\t"
                    f"{result['avg_fitness']:.4f}\t"
                    f"{result['max_fitness']:.4f}\t"
                    f"{result['total_fitness']:.4f}\n"
                )

def main():
    # Configuration
    ewca_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt"
    sedmtg_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/tmp/GO_weighted_STRING_humain.txt"
    known_complexes_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/STRING_humain.txt"
    output_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/detected_complexes_STRING_humain2.txt"
    metrics_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/metrics/metrics_STRING_humain2.tsv"
    
    # Create extractor
    extractor = ProteinComplexExtractor()
    
    # Load known complexes if available
    if os.path.exists(known_complexes_file):
        print("Loading known complexes...")
        extractor.load_known_complexes(known_complexes_file)
        print(f"Loaded {len(extractor.known_complexes)} known complexes")
    else:
        print("Warning: No known complexes file found at", known_complexes_file)
    
    # Build PPI graph from weighted network (for fitness calculations)
    print("Building PPI graph for fitness calculations...")
    extractor.build_ppi_graph(sedmtg_file)
    
    # Generate complexes and metrics
    print("\nGenerating protein complexes and calculating metrics...")
    extractor.generate_complexes(
        ewca_file=ewca_file,
        sedmtg_file=sedmtg_file,
        output_file=output_file,
        metrics_file=metrics_file
    )
    
    print("\nProcessing complete!")

if __name__ == "__main__":
    main()

Loading known complexes...
Loaded 1477 known complexes
Building PPI graph for fitness calculations...

Generating protein complexes and calculating metrics...

Running EWCA method...


EWCA progress:   0%|          | 0/8 [00:00<?, ?it/s]

Total number of proteins: 12397
Total number of interactions: 101652
